# 03 - Deep hedging sous Heston

Objectif: entra?ner rapidement VNN/SNN/RNN sur des trajectoires Heston. Les tailles sont r?duites pour une ex?cution de contr?le.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from deep_hedging import RNN, SNN, VNN, evaluate, evaluate_rnn, get_optimizer, training_loop, training_rnn
from models import compute_payoff_europeen, simulate_heston
from signatures import compute_signatures


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
I_train, I_test, J = 1_024, 512, 40
T, V0, K = 0.5, 0.0625, 1.0

S_train, V_train, dS_train, sigma_train, states_train = simulate_heston(1, V0, 4, V0, 0.7, -0.7, T, J, I_train, seed=1)
S_test, V_test, dS_test, sigma_test, states_test = simulate_heston(1, V0, 4, V0, 0.7, -0.7, T, J, I_test, seed=2)

payoff_train = compute_payoff_europeen(S_train, K)
payoff_test = compute_payoff_europeen(S_test, K)


In [ ]:
train_ds = TensorDataset(
    torch.tensor(states_train, dtype=torch.float32),
    torch.tensor(dS_train, dtype=torch.float32),
    torch.tensor(payoff_train, dtype=torch.float32),
)
test_ds = TensorDataset(
    torch.tensor(states_test, dtype=torch.float32),
    torch.tensor(dS_test, dtype=torch.float32),
    torch.tensor(payoff_test, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)


In [ ]:
model_vnn = VNN(input_features=3, output_features=1).to(device)
optimizer, scheduler = get_optimizer(model_vnn)
training_loop(model_vnn, train_loader, 64, optimizer, scheduler, num_epochs=2, device=device)
hedging_pnl, mse, mean_pnl = evaluate(model_vnn, test_loader, device)
print('VNN X0:', float(model_vnn.X0.detach().cpu()))
print('VNN MSE:', float(mse.detach().cpu()), 'Mean PnL:', float(mean_pnl.detach().cpu()))


In [ ]:
try:
    sig_train = compute_signatures(states_train, depth=3)
    sig_test = compute_signatures(states_test, depth=3)
    sig_train_ds = TensorDataset(torch.tensor(sig_train, dtype=torch.float32), torch.tensor(dS_train, dtype=torch.float32), torch.tensor(payoff_train, dtype=torch.float32))
    sig_test_ds = TensorDataset(torch.tensor(sig_test, dtype=torch.float32), torch.tensor(dS_test, dtype=torch.float32), torch.tensor(payoff_test, dtype=torch.float32))
    print('SNN input dimension:', sig_train.shape[-1])
except ImportError as exc:
    print(exc)
